# AEGIS-0.5M: High-Subsonic (Mach 0.5) Autonomous Surveillance UAV Digital Twin

Self-contained interactive research notebook investigating:
1. **Subsonic Compressible Aerodynamics** (Prandtl-Glauert $\beta = \sqrt{1-M^2}$, $M = 0.50 \approx 171.5\text{ m/s}$)
2. **2-Axis Active Gimbal Motion-Blur Cancellation** ($\dot{\theta} = v \cos^2\phi / H$, maintaining $<0.5\text{ px}$ smear)
3. **20-Floor Building Optical Precision** ($\text{GSD} = 0.98\text{ cm/px}$ at $H = 65\text{ m}$, Johnson Criteria)
4. **Cross-Modal RGB + LWIR ($8\text{–}14\ \mu\text{m}$) Bayesian Dempster-Shafer Fusion & Behavioral Alerts**
5. **Zero-Dependency C++23 Edge Pipeline Benchmark** ($<2.5\text{ ms}$ frame latency)

In [ ]:
import os, sys, pathlib
if pathlib.Path('src').exists():
    ROOT = pathlib.Path('.').resolve()
elif pathlib.Path('../src').exists():
    ROOT = pathlib.Path('..').resolve()
else:
    ROOT = pathlib.Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Workspace root:', ROOT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.physics.aerodynamics import AerodynamicModel, load_aero_parameters_from_yaml

params = load_aero_parameters_from_yaml('configs/uav_mission_profile.yaml')
aero = AerodynamicModel(params)
cruise = aero.evaluate_flight_state(mach=0.50, altitude_m=65.0, off_nadir_deg=0.0)

print(f"Cruise Mach        : {cruise.mach:.2f} ({cruise.airspeed_mps:.1f} m/s)")
print(f"Prandtl-Glauert β  : {cruise.beta_pg:.4f}")
print(f"Compressible CD    : {cruise.cd_total:.5f}")
print(f"Required Thrust    : {cruise.thrust_required_n:.2f} N")
print(f"Gimbal Rate @ 65 m : {cruise.gimbal_angular_rate_deg_s:.2f} deg/s")
print(f"Motion Smear       : {cruise.uncompensated_smear_px:.2f} px (uncomp) -> {cruise.stabilized_smear_px:.3f} px (FMC)")

sweep = aero.sweep_mach_envelope()
m_vals = [s.mach for s in sweep]
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(m_vals, [s.uncompensated_smear_px for s in sweep], '--', color='#8B959E', label='Uncompensated Smear (px)')
ax.plot(m_vals, [s.stabilized_smear_px for s in sweep], color='#750014', lw=2.3, label='2-Axis Active FMC Smear (px)')
ax.axhline(0.5, color='#FF1423', ls=':', label='0.50 px Optical Limit')
ax.set_xlabel('Mach Number'); ax.set_ylabel('Pixel Smear'); ax.set_title('Mach 0.05-0.55 Motion Blur Suppression')
ax.legend(); ax.grid(alpha=0.3); plt.show()

In [ ]:
from src.physics.optics_gsd import OpticalGSDModel, load_optics_config_from_yaml

optics = OpticalGSDModel(load_optics_config_from_yaml('configs/uav_mission_profile.yaml'))
gsd_65 = optics.evaluate_altitude(altitude_m=65.0, off_nadir_deg=0.0, residual_smear_px=cruise.stabilized_smear_px)

print(f"Altitude AGL       : {gsd_65.altitude_m:.1f} m (~{gsd_65.approx_building_floors:.0f} floors)")
print(f"Wide 16mm RGB GSD  : {gsd_65.gsd_nadir_cm_px:.4f} cm/px")
print(f"Human Target Span  : {gsd_65.human_width_px:.1f} x {gsd_65.human_height_px:.1f} px ({gsd_65.human_area_px:.0f} px^2)")
print(f"Tele 220mm GSD     : {gsd_65.telephoto_gsd_mm_px:.3f} mm/px (6mm Ant: {gsd_65.telephoto_micro_length_px:.1f} x {gsd_65.telephoto_micro_width_px:.1f} px)")
print(f"Johnson P(ID)      : {gsd_65.p_identification*100:.2f}%")

In [ ]:
import torch
from src.vision.multispectral_fusion import MultiSpectralSceneGenerator, CrossModalDSDetector

gen = MultiSpectralSceneGenerator(seed=2026)
scene = gen.generate_scene(regime='foliage_occlusion', num_humans=3, num_distractors=2)
detector = CrossModalDSDetector().eval()
with torch.no_grad():
    out = detector(scene.rgb, scene.lwir_kelvin)
boxes = detector.extract_detections(out, scene.lwir_kelvin, score_map_key='fused_prob')
print(f'Detected {len(boxes)} targets under foliage occlusion (GT={len(scene.gt_boxes)})')

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].imshow(scene.rgb.permute(1, 2, 0).numpy()); axes[0].set_title('RGB Optical (Occluded)')
axes[1].imshow(scene.lwir_kelvin[0].numpy(), cmap='inferno'); axes[1].set_title('LWIR 8-14 um (Kelvin)')
axes[2].imshow(out['fused_prob'][0, 0].numpy(), cmap='magma'); axes[2].set_title('Dempster-Shafer Fused Belief')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
from simulations.sim1_mach05_envelope import run_sim1
from simulations.sim2_altitude_gsd_precision import run_sim2
from simulations.sim3_rgb_lwir_occlusion import run_sim3
from simulations.sim4_urban_safety_alerts import run_sim4
from visualization.plot_figures import generate_all_figures

run_sim1(); run_sim2(); run_sim3(); run_sim4()
paths = generate_all_figures()
print('Generated 300-DPI figures:', [str(p) for p in paths])